In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Programing\CyberSec-Reasoner


In [3]:
import random
import warnings
from pprint import pprint
from datasets import load_dataset, load_from_disk, concatenate_datasets, DatasetDict

warnings.filterwarnings("ignore")

*****
# Download Huggingface Datasets and save to Raw folder

In [4]:
load_dataset("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.0").save_to_disk("data/raw/Cybersecurity-Dataset-Fenrir-v2.0")
load_dataset("Mohannadcse/cybersec-reasoning-merged").save_to_disk("data/raw/cybersec-reasoning-merged")
load_dataset("trendmicro-ailab/Primus-Reasoning").save_to_disk("data/raw/Primus-Reasoning")

Generating train split: 83920 examples [00:01, 42907.32 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 23146/23146 [00:00<00:00, 177429.36 examples/s]
Generating train split: 4891 examples [00:00, 96381.60 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 4891/4891 [00:00<00:00, 49093.23 examples/s]


*****
# Merge Fenrir & Cybersec datasets for SFT

In [4]:
fenrir = load_from_disk("data/raw/Cybersecurity-Dataset-Fenrir-v2.0")
print(fenrir)
cybersec = load_from_disk("data/raw/cybersec-reasoning-merged")
print(cybersec)

DatasetDict({
    train: Dataset({
        features: ['system', 'user', 'assistant'],
        num_rows: 83920
    })
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'reasoning_content', 'answer'],
        num_rows: 23146
    })
})


In [17]:
pprint(fenrir["train"][random.randint(0, len(fenrir["train"]) - 1)])

{'assistant': 'Monitoring strategies for detecting operational anomalies in '
              'metamorphic malware analysis using category theory frameworks '
              'require sophisticated approaches that address the dynamic and '
              'evolving nature of these threats. Traditional signature-based '
              'detection methods are insufficient due to the polymorphic '
              'transformations that metamorphic malware undergoes, altering '
              'its code structure while maintaining '
              'functionality.\\\\n\\\\n**Behavioral Analysis**: One effective '
              'strategy involves behavioral monitoring, which focuses on the '
              'actions performed by the malware rather than its static '
              "characteristics. This approach aligns with category theory's "
              'emphasis on morphisms and transformations, where the focus is '
              'on how objects (malware instances) interact within a category '
          

In [21]:
pprint(cybersec["train"][random.randint(0, len(cybersec["train"]) - 1)])

{'answer': "If you don't address this issue, your system could be at risk of "
           "being compromised by external changes to the tools you're relying "
           'on. This might allow unauthorized individuals to alter how your '
           'processes work, potentially leading to security breaches, data '
           'exposure, or unintended actions that could harm your operations.',
 'prompt': 'cwe_id:CWE-829\n'
           'cwe_name:Inclusion of Functionality from Untrusted Control Sphere\n'
           'affected_line:Unpinned Actions Full Length Commit SHA\n'
           'partial_code:- name: Docker meta\n'
           '        id: meta\n'
           '        uses: docker/metadata-action@v3\n'
           '        with:\n'
           '          images: |\n'
           '            lablabs/cloudflare_exporter\n'
           '            ghcr.io/lablabs/cloudflare_exporter\n'
           '          # generate Docker tags based on the following '
           'events/attributes\n'
       

In [5]:
def format_fenrir(data):
    messages = [
        {"role": "system", "content": data["system"]},
        {"role": "user", "content": data["user"]},
        {"role": "assistant", "content": data["assistant"]},
    ]
    return {"messages": messages}

In [ ]:
def format_cybersec_dataset(data, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data['prompt']},
        {"role": "assistant", "content": f"<|reasoning_start|>\n{data['reasoning_content']}\n<|reasoning_end|>\n\n{data['answer']}"},
    ]
    return {"messages": messages}

In [11]:
fenrir = fenrir.map(
    format_fenrir,
    remove_columns=fenrir["train"].column_names
)

system_prompt = ("You are a cybersecurity expert. Analyze the given vulnerability context carefully and "
                "think step by step to understand the root cause, risk, and impact. Then provide a clear, "
                "concise, and accurate answer to the user's question based on your reasoning.")

cybersec = cybersec.map(
    lambda x: format_cybersec_dataset(x, system_prompt),
    remove_columns=cybersec["train"].column_names
)

Map: 100%|██████████| 23146/23146 [00:01<00:00, 13728.39 examples/s]


In [12]:
pprint(cybersec["train"][1545])

{'messages': [{'content': 'You are a cybersecurity expert. Analyze the given '
                          'vulnerability context carefully and think step by '
                          'step to understand the root cause, risk, and '
                          'impact. Then provide a clear, concise, and accurate '
                          "answer to the user's question based on your "
                          'reasoning.',
               'role': 'system'},
              {'content': 'cwe_id:CWE-200\n'
                          'cwe_name:Exposure of Sensitive Information to an '
                          'Unauthorized Actor\n'
                          'affected_line:Exposure of Sensitive Information to '
                          'an Unauthorized Actor in SonarSource SonarQube API\n'
                          'partial_code:org.sonarsource.sonarqube:sonar-plugin-api '
                          '6.3\n'
                          'file_name:pom.xml\n'
                          'status:True P

In [13]:
dataset = concatenate_datasets([fenrir["train"], cybersec["train"]])
dataset

Dataset({
    features: ['messages'],
    num_rows: 107066
})

In [14]:
dataset = dataset.train_test_split(test_size=0.10, seed=42, shuffle=True)
split = dataset["test"].train_test_split(test_size=0.50, seed=42, shuffle=True)
dataset = DatasetDict({
    "train": dataset["train"],
    "validation": split["train"],
    "test": split["test"]
})

dataset.save_to_disk("data/processed/cybersecurity_sft_dataset")
print(dataset)

Saving the dataset (1/1 shards): 100%|██████████| 5354/5354 [00:00<00:00, 138977.76 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 96359
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5353
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 5354
    })
})


*****
# Prepare Primus Dataset

In [15]:
primus = load_from_disk("data/raw/Primus-Reasoning")
primus

DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 4891
    })
})

In [16]:
primus["train"][0]["messages"]

[{'content': "Analyze the following CVE description and map it to the appropriate CWE. Provide a brief justification for your choice. Ensure the last line of your response contains only the CWE ID.  CVE Description: It's possible to craft a request for appointment edit screen, which could lead to the XSS attack. This issue affects: OTRS AG ((OTRS)) Community Edition 6.0.x version 6.0.1 and later versions. OTRS AG OTRS 7.0.x version 7.0.28 and prior versions.",
  'role': 'user'},
 {'content': "<|reserved_special_token_0|>Okay, let's tackle this CVE analysis. The description mentions that crafting a request for the appointment edit screen could lead to an XSS attack. XSS, or Cross-Site Scripting, is a common vulnerability.\n\nFirst, I need to recall the different types of XSS. There's stored, reflected, and DOM-based. The CVE doesn't specify where the input is injected, just that a crafted request can cause it. Since it's about the appointment edit screen, maybe user input isn't properly

In [28]:
def format_primus(data, system_prompt):
    data = data["messages"]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data[0]["content"]},
        {"role": "assistant", "content": (data[1]["content"]).replace("<|reserved_special_token_0|>", "<|reasoning_start|>\n").replace("<|reserved_special_token_1|>", "<|reasoning_end|>")},
    ]
    return {"messages": messages}

In [31]:
system_prompt = """You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.

Your task is to analyze CVE descriptions and accurately map them to the most appropriate CWE (Common Weakness Enumeration) category.

You must:
- Carefully analyze the vulnerability description
- Identify the underlying vulnerability type
- Determine the root cause of the weakness
- Ensure your reasoning logically supports the final CWE classification

Focus on:
- Root cause analysis rather than surface-level patterns
- Using precise and relevant security concepts
- Maintaining clear, consistent, and logically sound reasoning

Avoid:
- Guessing without justification
- Contradictions between reasoning and conclusion
- Overly generic or irrelevant explanations

Your final answer must always be fully supported by your reasoning."""

In [34]:
primus = primus.map(
    lambda x: format_primus(x, system_prompt),
    remove_columns=primus["train"].column_names
)

primus.save_to_disk("data/processed/primus_reasoning_dataset")
print(primus)

Saving the dataset (1/1 shards): 100%|██████████| 4891/4891 [00:00<00:00, 348421.16 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 4891
    })
})


In [36]:
primus["train"][random.randint(0, len(primus["train"]) - 1)]

{'messages': [{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE descriptions and accurately map them to the most appropriate CWE (Common Weakness Enumeration) category.\n\nYou must:\n- Carefully analyze the vulnerability description\n- Identify the underlying vulnerability type\n- Determine the root cause of the weakness\n- Ensure your reasoning logically supports the final CWE classification\n\nFocus on:\n- Root cause analysis rather than surface-level patterns\n- Using precise and relevant security concepts\n- Maintaining clear, consistent, and logically sound reasoning\n\nAvoid:\n- Guessing without justification\n- Contradictions between reasoning and conclusion\n- Overly generic or irrelevant explanations\n\nYour final answer must always be fully supported by your reasoning.',
   'role': 'system'},
  {'content': 'Analyze the following CVE description and map it to the appropriate CWE. Provi